[Guide Index](README.md)


# 04 SHACL 1.2

`starlayer.shacl` is a SHACL engine written against the SHACL 1.2 draft specifications (Core, SPARQL Extensions, Node Expressions, Inference Rules). 

This guide introduces the`StarShaclShema` class, used for managing various SHACL functions, and discusses how the engine handles RDF 1.2 graphs (triple terms, direction-tagged literals) and serveral other SHACL 1.2 enhacements.  

Additional guides are provided to explore more in-depth SHACL 1.2 functionality supported by StarLayer.


- **[SHACL node expressions](04a-shacl-node-expressions.ipynb)** — the `shnex:`/`sparql:` node-expression vocabulary, `sh:values`, custom functions.
- **[SHACL and SPARQL](04b-shacl-and-sparql.ipynb)** — `sh:sparql` constraints, user-defined `sh:ConstraintComponent`.
- **[SHACL inference rules](04c-shacl-inference-rules.ipynb)** — `sh:rule`/`sh:TripleRule`/`sh:SPARQLRule`, execution ordering, rule sets, provenance.
- **[SHACL UI](04d-shacl-ui.ipynb)** — `shui:` presentation metadata.
- **[SHACL profiling](04e-shacl-profiling.ipynb)** — `sh:ShapesGraph`/`sh:DataGraph`/`owl:imports` packaging conventions, profiling with SHACL and profiling of SHACL.
- **[SHACL subgraph extraction ](04f-shacl-subgraph-extraction.ipynb)** — `extract_subgraph()`, `close_shape()`.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.


In [1]:
from starlayer.graph import Namespace
from starlayer.graph import StarLayerGraph
from starlayer.shacl import StarShaclSchema

# EX and SH are starlayer.graph Namespaces: their terms are pyoxigraph NamedNodes, used to look terms up in the results below
# (StarLayerGraphs, see section 1.1); the StarLayerGraph inputs are built by parsing Turtle.
EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

## 1. What `starlayer.shacl` offers






`StarShaclSchema` is `starlayer.shacl`'s own SHACL entry point — construct it with `StarShaclSchema(shacl_graph=...)`, then call `.validate(data_graph=...)` (or one of the other two processing modes below) to get a result back. If you have validated with a SHACL library before (see [Getting Started](01-getting-started.ipynb)), `StarShaclSchema` is the same idea, extended:

- Validation over RDF 1.2 graphs — triple terms, reification, direction-tagged literals
- SHACL rule support (`sh:rule`, `sh:TripleRule`, `sh:SPARQLRule`, rule sets)
- Direction-aware datatype constraints
- Fine-grained, per-constraint severity and messages, including new severity levels (`sh:Debug`, `sh:Trace`)
- SHACL 1.2 Node Expressions (`shnex:` combinators, `sparql:`-namespaced SPARQL built-ins, and user-defined custom node expression functions)
- A standalone, `cbd()`-like utility function, `extract_subgraph()` - not a `StarShaclSchema` method, but pulls exactly the subgraph a shape covers for a focus node, or across every entry point a whole shapes graph declares; see the [subgraph extraction guide](04f-shacl-subgraph-extraction.ipynb)

### 1.1 Graphs in, graphs out

The data graph, shapes graph and ontology graph can be `StarLayerGraph` or `StarLayerDataset` objects (see the [Oxigraph wrapper guide](02d-oxigraph-wrapper.ipynb)). `starlayer.shacl` accepts them directly. When the inputs are `StarLayerGraph` or `StarLayerDataset` objects (or a `pyoxigraph.Store`), the results are `StarLayerGraph` objects: `result.report_graph`, `RulesResult.inferred_graph` and the graph returned by `evaluate()` hold pyoxigraph terms and carry only the prefixes of your input. The examples below look results up with `EX.claim` style terms (a `starlayer.graph.Namespace` gives them) and read values with `.value`, for example `int(v.value)`. The input graphs are never changed.

## 2. Three processing modes

`StarShaclSchema` exposes three independent processing modes over the same data/shapes graph pair.

### 2.1 `validate()` — conformance checking, never mutates

The familiar SHACL entry point: given a data graph and a shapes graph, report whether the data conforms. The rest of this guide's examples (sections 3-4 below) are all `validate()` examples, showing RDF-1.2-specific and SHACL-1.2-specific behavior.

### 2.2 `apply_rules()` — executes `sh:rule`, materializes real triples

Runs every `sh:rule` in the shapes graph to a fixed point and returns the result merged into a *new* data graph — the original `data_graph` is never mutated. Full depth (execution ordering, rule sets, condition filtering, provenance) is in the [inference rules guide](04c-shacl-inference-rules.ipynb); here's the shortest possible example: a rule that adds `ex:trusted ex:yes` to any node reified with `ex:confidence "high"`.

In [2]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> ;
      ex:confidence "high" .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
        @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:ConfidenceRule a sh:NodeShape ;
      sh:targetSubjectsOf ex:confidence ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject [ shnex:var "focusNode" ] ;
        sh:predicate ex:trusted ;
        sh:object ex:yes ;
        sh:condition [ sh:property [ sh:path ex:confidence ; sh:hasValue "high" ] ] ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print("new triple materialized:", (EX.claim, EX.trusted, EX.yes) in result.inferred_graph)
print("conforms:", result.validation.conforms)

new triple materialized: True
conforms: True


### 2.3 `evaluate()` — computed properties for display, without storing them

A third, independent mode: compute every `sh:values`-declared *virtual* property and return it merged into a throwaway copy of the data graph, without ever writing to the real one. Useful for "display or query this as if it were stored" without actually persisting a derived value. Full depth (including what happens when a stored value already exists for the same property) is in the [node expressions guide](04a-shacl-node-expressions.ipynb#3.-Computed-properties:-sh:values); here, `ex:friendCount` is computed on demand rather than stored.

In [3]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:friend ex:bob, ex:carol .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .

    ex:S a sh:NodeShape ; sh:targetNode ex:alice ;
      sh:property [ sh:path ex:friendCount ;
                    sh:values [ shnex:count [ shnex:pathValues ex:friend ] ] ] .
""", format="turtle")

display_result = StarShaclSchema(shacl_graph=shapes).evaluate(data_graph=data)
print("friend count:", [int(v.value) for v in display_result.objects(EX.alice, EX.friendCount)])
print("original data_graph still has zero ex:friendCount triples:",
      len(list(data.triples((None, EX.friendCount, None)))) == 0)

friend count: [2]
original data_graph still has zero ex:friendCount triples: True


### Related: `extract_subgraph()`, a standalone utility (not a processing mode)

Given a focus node that conforms to a shape - or every entry point a whole shapes graph declares - extracts exactly the subgraph of *real, stored* triples that shape's constraints actually covered - not the whole graph around it. Conceptually `StarLayerGraph.cbd()`, but shape-driven: a plain function, not a `StarShaclSchema` method. The motivating use case (hash a commitment, later re-verify it hasn't changed) and the full mechanics are in the [subgraph extraction guide](04f-shacl-subgraph-extraction.ipynb).

## 3. SHACL 1.2 node kinds and severities

### 3.1 Is a value a triple term? `sh:nodeKind ( sh:TripleTerm )`

The list-valued form of `sh:nodeKind` recognizes `sh:TripleTerm` as one of its choices, letting a shape require that a value — e.g. the object of `rdf:reifies` — is genuinely an RDF 1.2 triple term, not a plain URI, blank node, or literal. The engine checks every `sh:nodeKind` value itself, in the list form or as a bare scalar: `sh:TripleTerm`, `sh:IRI`, `sh:BlankNode`, `sh:Literal`, or a combined kind like `sh:BlankNodeOrIRI`.

In [4]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    [] a sh:NodeShape ;
      sh:targetSubjectsOf rdf:reifies ;
      sh:property [ sh:path rdf:reifies ; sh:nodeKind ( sh:TripleTerm ) ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .
""", format="turtle")
result = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data, meta_shacl=False)
print("conforms (real triple term):", result.conforms)

bad_data = StarLayerGraph()
bad_data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies ex:not_a_triple_term .
""", format="turtle")
result_bad = StarShaclSchema(shacl_graph=shapes).validate(data_graph=bad_data, meta_shacl=False)
print("conforms (plain URI, not a triple term):", result_bad.conforms)

conforms (real triple term): True
conforms (plain URI, not a triple term): False


### 3.2 New severity levels: `sh:Debug`, `sh:Trace`

SHACL 1.2 adds two severities below `sh:Info`/`sh:Warning`. Unlike those two (which need `allow_warnings=True`/`allow_infos=True` to stop blocking `conforms`), a `sh:Debug`/`sh:Trace` result **never** blocks conformance — it's recorded in the report but has no effect on `result.conforms`, useful for constraints you want visibility into without failing validation over.

In [5]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:age ; sh:minCount 1 ; sh:severity sh:Debug ] .
""", format="turtle")
result = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data)
print("conforms despite missing ex:age:", result.conforms)  # True - sh:Debug never blocks
print(result.report_text)

conforms despite missing ex:age: True
Validation Report
Conforms: True
Results (1):
Validation Result in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Debug
	Source Shape: [ sh:minCount 1 ; sh:path ex:age ; sh:severity sh:Debug ]
	Focus Node: ex:alice
	Result Path: ex:age
	Message: Needs at least 1 value for ex:age



### 3.3 Fine-grained severity via reification: `{| sh:severity ... |}`

A single constraint-value triple can carry its own `sh:severity`/`sh:deactivated` override via RDF 1.2's inline annotation shorthand — distinct from, and finer-grained than, a shape's own `sh:severity`. Currently wired for `sh:datatype` (severity) and `sh:property` (deactivation), and only on a constraint declared directly on the targeted shape (not yet inside a nested `sh:property [...]` blank node).

In [6]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:AgeMustBeIntShape a sh:NodeShape ;
      sh:targetNode ex:alice ;
      sh:datatype xsd:integer {| sh:severity sh:Warning |} .
""", format="turtle")
data = StarLayerGraph()
data.add((EX.alice, EX.dummy, EX.alice))  # ex:alice is a URI, not an xsd:integer literal - violates

result = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data, meta_shacl=False)
severities = {result.report_graph.qname(o) for _, _, o in
              result.report_graph.triples((None, SH.resultSeverity, None))}
print("severity from the annotation:", severities)  # {'sh:Warning'}, not the default sh:Violation
print("conforms:", result.conforms)                  # still False - Warning still blocks unless allowed

result2 = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data, meta_shacl=False, allow_warnings=True)
print("conforms with allow_warnings=True:", result2.conforms)

severity from the annotation: {'sh:Warning'}
conforms: False
conforms with allow_warnings=True: True


### 3.4 Direction-aware datatype constraints

`rdf:dirLangString` (the datatype of a direction-tagged literal such as `"hello"@en--ltr`) works as an ordinary `sh:datatype` value.

In [7]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:n1 a ex:Note ;
      ex:label "hello"@en--ltr .
""", format="turtle")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:LangShape a sh:NodeShape ;
      sh:targetClass ex:Note ;
      sh:property [ sh:path ex:label ; sh:datatype rdf:dirLangString ] .
""", format="turtle")
result = StarShaclSchema(shacl_graph=shapes).validate(data_graph=data)
print("conforms:", result.conforms)

conforms: True


## 4. Node expressions

SHACL 1.2 Node Expressions — the `shnex:` combinator vocabulary, the `sparql:` namespace exposing SPARQL built-ins, and user-defined custom node expression functions (including "named node expressions") — have their own dedicated guide: **[SHACL node expressions](04a-shacl-node-expressions.ipynb)**.

## 5. Reading a result, and what goes wrong

`validate()` returns a `ValidationResult`. `conforms` says whether the data conforms; `report_graph` is the SHACL validation report, a `StarLayerGraph` with one `sh:ValidationResult` for each violation (read `sh:focusNode`, `sh:resultPath`, `sh:sourceConstraintComponent`, `sh:sourceShape`, `sh:resultSeverity`, `sh:value` and `sh:resultMessage` from it, not the message text); `report_text` is the same report as text for a person; `diagnostics` is an `ExecutionDiagnostics` with the counters of the run (`report_triples`, `recursion_revisits`, ...). The message of a result is processor-defined, so code that decides something reads the structured fields.

In [8]:
from starlayer.graph import StarLayerGraph
from starlayer.shacl import SH, validate

shapes_r = StarLayerGraph()
shapes_r.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:PersonShape a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ; sh:datatype xsd:string ] .
""", format="turtle")
data_r = StarLayerGraph()
data_r.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
    ex:bob a ex:Person ; ex:name "Bob" .
""", format="turtle")

result = validate(data_r, shapes_r)
print("conforms:", result.conforms, "| report triples:", result.diagnostics.report_triples)
for violation in result.report_graph.objects(None, SH.result):
    print(" focus node:", result.report_graph.value(violation, SH.focusNode).value,
          "| component:", result.report_graph.value(violation, SH.sourceConstraintComponent).value.rsplit("#", 1)[-1],
          "| path:", result.report_graph.value(violation, SH.resultPath).value)
print(result.report_text)

conforms: False | report triples: 13
 focus node: http://example.org/alice | component: MinCountConstraintComponent | path: http://example.org/name
Validation Report
Conforms: False
Results (1):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:string ; sh:minCount 1 ; sh:path ex:name ]
	Focus Node: ex:alice
	Result Path: ex:name
	Message: Needs at least 1 value for ex:name



### 5.1 Each graph of a dataset

SHACL validates one data graph, and `validate()` sees a `StarLayerDataset` as the union of its graphs. `validate_each()` validates the graphs of a dataset (or a list or mapping of graphs) one by one against the same shapes, parsing the shapes once, and returns `{graph name: ValidationResult}` (`None` is the default graph, when it holds triples).

In [9]:
from starlayer.graph import StarLayerDataset
from starlayer.shacl import validate_each

people = StarLayerDataset(default_union=False)
people.update("""INSERT DATA {
  GRAPH <urn:team:a> { <http://example.org/ann> a <http://example.org/Person> ; <http://example.org/name> "Ann" }
  GRAPH <urn:team:b> { <http://example.org/cy> a <http://example.org/Person> } }""")
for name, each in validate_each(people, shapes_r).items():
    print(name, "conforms" if each.conforms else "does not conform")

<urn:team:a> conforms
<urn:team:b> does not conform


### 5.2 When the shapes are wrong

A shapes graph that is malformed raises rather than being skipped. `ShapeError` is a shape that cannot be read (a `sh:NodeShape` with a `sh:path`, a `sh:property` value with no `sh:path`, a bad regular expression); `RuleError` is the same for rules; `MetaShaclError` is raised when `meta_shacl=True` and the shapes graph does not conform to SHACL's own shapes for shapes (it is off by default, because it costs about as much as the validation itself); `EvaluationFailure` is a node expression with no result for the data, which is not the same as an empty result. `ShapeError`, `RuleError` and `MetaShaclError` are `ValueError`s, so one `except ValueError` covers them; `EvaluationFailure` is a plain `Exception`.

In [10]:
from starlayer.shacl import MetaShaclError, ShapeError

misplaced = StarLayerGraph()
misplaced.parse(data="@prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . ex:S a sh:NodeShape ; sh:path ex:p .", format="turtle")
try:
    validate(data_r, misplaced)
except ShapeError as error:
    print("ShapeError:", error)

wrong_value = StarLayerGraph()
wrong_value.parse(data="""@prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:S a sh:NodeShape ; sh:targetClass ex:Person ; sh:minCount "many" .""", format="turtle")
try:
    validate(data_r, wrong_value, meta_shacl=True)
except MetaShaclError as error:
    print("MetaShaclError:", str(error).splitlines()[0])

ShapeError: http://example.org/S: a sh:NodeShape cannot be the subject of sh:path
MetaShaclError: SHACL File does not validate against the SHACL Shapes SHACL (MetaSHACL) file.


### 5.3 Entailment during validation

Which nodes a shape targets, and whether a value is of a class, follow `rdfs:subClassOf` in the data without any setting: SHACL defines it so. Anything more, such as a node being an `ex:Employee` because it has a property whose `rdfs:domain` is `ex:Employee`, needs an entailment regime. Ask for one per call with `inference=ENTAILMENT.RDFS` (a regime IRI, as `infer()` takes), or state it in the shapes graph with `sh:entailment` on a shape, so the shapes carry what they need. `target_nodes(data_graph=, shacl_graph=, shape_node=, inference=)` returns the nodes a shape targets: the nodes `validate()` would check.

In [11]:
from starlayer.graph import ENTAILMENT, NamedNode
from starlayer.shacl import target_nodes

staff_shapes = StarLayerGraph()
staff_shapes.parse(data="""
    @prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:EmployeeShape a sh:NodeShape ; sh:targetClass ex:Employee ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")
staff = StarLayerGraph()
staff.parse(data="""
    @prefix ex: <http://example.org/> . @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:worksAt rdfs:domain ex:Employee .
    ex:alice ex:worksAt ex:Acme .
""", format="turtle")
employee_shape = NamedNode("http://example.org/EmployeeShape")

print("targets, no entailment:", [n.value for n in target_nodes(data_graph=staff, shacl_graph=staff_shapes, shape_node=employee_shape)])
print("targets, inference=RDFS:", [n.value for n in target_nodes(data_graph=staff, shacl_graph=staff_shapes, shape_node=employee_shape, inference=ENTAILMENT.RDFS)])
print("conforms without:", validate(staff, staff_shapes).conforms, "| with:", validate(staff, staff_shapes, inference=ENTAILMENT.RDFS).conforms)

staff_shapes.add((employee_shape, SH.entailment, ENTAILMENT.RDFS))  # or say it in the shapes graph
print("conforms with sh:entailment in the shapes:", validate(staff, staff_shapes).conforms)

targets, no entailment: []
targets, inference=RDFS: ['http://example.org/alice']
conforms without: True | with: False
conforms with sh:entailment in the shapes: False


## 6. Deriving shapes from data

`graph.derive_shape()` writes a shapes graph from the data itself: one `sh:NodeShape` for each `rdf:type` found (never per node), with a property shape for each property its instances use. From the values it finds, it sets `sh:minCount` and `sh:maxCount`, `sh:datatype` or `sh:class` or `sh:nodeKind` (a list when the kinds are mixed, `sh:TripleTerm` included), and the numeric and string ranges the data supports. The shape is `sh:closed`, with `sh:ignoredProperties` for `rdf:type` and the usual descriptive properties (`ignored_properties=` adds more, `use_default_ignored_properties=False` drops the defaults). The result is validated against the same data before it is returned, so it always describes it. Given a `template_shape`, it reconciles that shape with the data: constraints the data now breaks are recomputed from it, those with no data-driven fix (`sh:pattern`, comparisons between properties) are dropped with an `rdfs:comment`, and constraints it cannot judge are left alone and commented.

In [8]:
import warnings

from starlayer.shacl import validate

observed = StarLayerGraph()
observed.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:a a ex:Person ; ex:name "Ann" ; ex:age 30 ; ex:knows ex:b .
    ex:b a ex:Person ; ex:name "Bob" ; ex:age 41 .
""", format="turtle")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the fidelity warning (guide 2.d) does not matter here
    derived = observed.derive_shape()
    conforms = validate(observed, derived).conforms
print(derived.serialize(format="turtle", pretty=True, only_used=True))
print("the data conforms to its own derived shape:", conforms)

@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

_:b0 a sh:NodeShape ;
    sh:closed true ;
    sh:ignoredProperties (
        <http://purl.org/dc/terms/created>
        <http://purl.org/dc/terms/modified>
        rdf:type
        <http://www.w3.org/2000/01/rdf-schema#comment>
        <http://www.w3.org/2000/01/rdf-schema#label>
    ) ;
    sh:property [
        a sh:PropertyShape ;
        sh:datatype xsd:string ;
        sh:maxCount 1 ;
        sh:minCount 1 ;
        sh:path ex:name
    ], [
        a sh:PropertyShape ;
        sh:datatype xsd:integer ;
        sh:maxCount 1 ;
        sh:minCount 1 ;
        sh:path ex:age
    ], [
        a sh:PropertyShape ;
        sh:class ex:Person ;
        sh:maxCount 1 ;
        sh:path ex:knows
    ] ;
    sh:targetClass ex:Person .

the data conforms to its own derived shape: True


## Known limitations

The project tracks this explicitly against the live W3C drafts in [shacl12-gap-matrix.md](../../docs/design/shacl/shacl12-gap-matrix.md). As of this guide's last review:

- **Content-level validation of `sh:select`/`sh:ask`/`sh:construct` query *text*** — checked structurally today (cardinality, datatype), not parsed as real SPARQL. A deliberate scope decision, not pursued.
- **SHACL 1.2 UI's `shui:WidgetScore`/`shui:WidgetAcceptMatcher` widget-selection algorithm** — `shui:` annotations on shapes validate fine (see [SHACL UI](04d-shacl-ui.ipynb)); the selection procedure itself is a genuinely new, optional feature, not built.
- **Calling a custom node expression function as an ordinary SPARQL function by name** from `sh:select`/`sh:sparqlExpr` query text (e.g. `ex:instanceCount(ex:Name)`) — the *node-expression* call forms of custom functions are supported today (see [SHACL node expressions](04a-shacl-node-expressions.ipynb)); this query-text form is a separate, larger mechanism (registering the function as a real SPARQL engine function).
- **Deferred**: SHACL's own Compact Syntax. (SRL (SPARQL Rules) is no longer deferred — see [SRL (SPARQL Rules)](03b-srl-sparql-rules.ipynb), a separate, SHACL-independent rules language.)
- ~~Deferred: `sh:PropertyRule`~~ — the construct no longer exists in the spec (confirmed absent from both `shacl12-inference-rules` and `shacl12-core`'s current text as of a 2026-09-28 re-check); an unrecognized rule type still correctly raises `RuleError`, kept as regression coverage rather than evidence of a real gap.

Already done, despite being listed as "planned" in earlier drafts of this guide: `sh:RuleSet`/`sh:hasRule`/`sh:includesRuleSet` (named rule subsets — `apply_rules(..., rule_set=<IRI>)`), `sh:resultAnnotation`/`sh:annotationProperty`/`sh:annotationVarName`/`sh:annotationValue` (for plain `sh:sparql` constraints), and SHACL 1.2 Profiling (investigated in full — no validator runtime behavior gap exists; see [SHACL profiling](04e-shacl-profiling.ipynb)).